### 1. Data Scraping

In [ ]:
"""
GOALS Project - Full FBref Scraper
====================================
Scrapes all stat types for Premier League, La Liga, Bundesliga
across 4 seasons and saves as JSON files.

BEFORE RUNNING:
1. Open https://fbref.com in Edge
2. F12 → Network → refresh → click first request → Headers
3. Copy the 'cookie:' value and paste it into COOKIES below

pip install requests beautifulsoup4 lxml pandas
"""

import cloudscraper
from bs4 import BeautifulSoup, Comment
import pandas as pd
import time
import os

OUTPUT_DIR = "data/raw"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# ── PASTE YOUR FRESH COOKIES HERE ───────────────────────────────────────────────────────────────
# ── REPLACE WITH NEW ONE IF YOU GET 403 ERRORS IN TEST RUN ──────────────────────────────────────
COOKIES = """srcssfull=yes; osano_consentmanager_uuid=ff6a2741-0a02-45f2-83ea-25e14ed5efd7; _ga=GA1.1.1005045719.1772577371; _pubcid=2c0ed4af-b1a1-49f6-8eab-98b2b5767519; _lc2_fpi=6fbc725567fc--01kjtxgdg68bz744rw8m32trqc; _lc2_fpi_meta=%7B%22w%22%3A1772577371654%7D; _lr_env_src_ats=false; ccuid=35c0d1ae-1b86-45ce-a99e-728d6077fe58; _cc_id=8e237d4706893798d6d890d956e93d; panoramaId_expiry=1773182171226; panoramaId=d98f5d55b0188f692de1e4385b42185ca02cc17e2b3366a813f584f959934c8b; panoramaIdType=panoDevice; gamera_user_id=9a74a339-f25f-4e0f-8955-8f8f13f0d534; 33acrossIdTp=EmhkvnIsuWQVgaDUvXh%2BIvoP0GG8ZTn6qhOIIYIDn4w%3D; _au_1d=AU1D-0100-001772577372-5LV5026I-S4P0; hubspotutk=6e1f4c43a8363a074b00b98d13271f7a; is_live=true; _li_dcdm_c=.fbref.com; _lr_retry_request=true; __hstc=218152582.6e1f4c43a8363a074b00b98d13271f7a.1772577395099.1772580491206.1772583253475.3; __hssrc=1; sr_n=1%7CWed%2C%2004%20Mar%202026%2000%3A14%3A29%20GMT; _ga_T897NZ0GWZ=GS2.1.s1772583219$o3$g1$t1772583598$j60$l0$h0; _ga_80FRT7VJ60=GS2.1.s1772583219$o3$g1$t1772583598$j60$l0$h0; cto_bidid=sQHjuV91VnZ2dTdiMmVsaXpxUEFuaTJLV0Rhc0xZVHNXN0NraTZNT0slMkZKSlkyVTdWWklEMUJxRTFzYTN0aTElMkJkRlpaSG1Lck5sdWZKRjZHY3hXZ1FST3h5SXJEcTRpcDI5MkxjQ3N4UlY0aWdFRWclM0Q; cto_dna_bundle=EP9zU190dmZMVlE4UHY4VlNBJTJCSUJVS1VCJTJCZHdCQXNKSVBpcnRnY3lTQnklMkJ2ZXR6WU9kZXlCQUVubGhKTDJEZjJSZTQxb1J4Z3RSOUJ1bUZGWXhBamRnTDhCZyUzRCUzRA; cto_bundle=-8CUrF90dmZMVlE4UHY4VlNBJTJCSUJVS1VCJTJCZjdPdFIlMkZ5NEFFVVh0QlJjbiUyQiUyQm53TWVLbEFYNzBhVldvS0RvUUE3OEdONkNRd2RsS2ZxbnlvSTR5VVk0QW5aTklzNE84ZFB0eHJXTTBGZVklMkI4ZGF6ZGUzMXpyN1NKTnM4dkNQQ2c4UTVpSlI2ZWFQRzNFWmYwRCUyQnVaM29yTGhKdyUzRCUzRA; __gads=ID=840e5941a11adf81:T=1772577372:RT=1772586057:S=ALNI_MZvPNXQOEXcb94kVXHpHsOcWjc44Q; __eoi=ID=0188330c55d6ea67:T=1772577372:RT=1772586057:S=AA-AfjbbUKoF_808OymoFLiQ4-VO; __cf_bm=zPx5vMKQW7ktan6dJzEqa.SvIaxthfN_baP7OwaXL.k-1772586058-1.0.1.1-yafR8CWDQB.qLDz3UbWcGR37Um4ZqjkVTCI._HszjCmLd9PbBnGhhZVqHbT9l6r3w3FRW2Mxrs5pNt3ysxZUYRVWQFBM8T91Zp8RJ_Ufv4E; cf_clearance=ZXaptvwo6wVlZ1IQrwI7YnkqYUTnfgjZQhUPovTy0W0-1772586063-1.2.1.1-BI0thC_TAMnK7mTg4ADL_KTEFlVKDI.io0m40AQ4Te.cQrYYh6gGBVPQqKCDmapUzDwX8a5fNTKcL6UPQmOmcMjAojHnQ_C99ZTujUXQnUpjXep7eDYOc7fPAiR5jTTzstcHaOoikbHgk9EaPn5_KBp1GZSkgAcZr69MJ0emtfcxWfEwxS_tqk2LylHI2oM.JAys.HxTanSlnrATWwTA1aVnGMUMNY1vixtyCc0B65c"""

# ── Replace User-Agent, Sec-Ch-Ua, and Sec-Ch-Ua-Platform with your own computer settings ───────
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/145.0.0.0 Safari/537.36 Edg/145.0.0.0",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7",
    "Accept-Language": "en-US,en;q=0.9,en-GB;q=0.8",
    "Accept-Encoding": "gzip, deflate, br, zstd",
    "Referer": "https://fbref.com/",
    "Sec-Ch-Ua": '"Not:A-Brand";v="99", "Microsoft Edge";v="145", "Chromium";v="145"',
    "Sec-Ch-Ua-Mobile": "?0",
    "Sec-Ch-Ua-Platform": '"Windows"',
    "Sec-Fetch-Dest": "document",
    "Sec-Fetch-Mode": "navigate",
    "Sec-Fetch-Site": "same-origin",
    "Upgrade-Insecure-Requests": "1",
}

# ── Create cloudscraper session with cookies ─────────────────────────────────────────────────────
scraper = cloudscraper.create_scraper(
    browser={"browser": "chrome", "platform": "windows", "desktop": True}
)
scraper.headers.update(HEADERS)

# Parse cookie string into session cookies
for cookie in COOKIES.split("; "):
    if "=" in cookie:
        name, value = cookie.split("=", 1)
        scraper.cookies.set(name.strip(), value.strip(), domain=".fbref.com")


In [20]:
# ── Leagues, seasons, stat types ──────────────────────────────
LEAGUES = {
    "premier_league": {"comp_id": "9", "name": "Premier-League"},
    "la_liga": {"comp_id": "12", "name": "La-Liga"},
    "bundesliga": {"comp_id": "20", "name": "Bundesliga"},
}

SEASONS = ["2021-2022", "2022-2023", "2023-2024", "2024-2025"]

STAT_TYPES = {
    "standard":           "stats",
    "goalkeeping":        "keepers",
    "shooting":           "shooting",
    "playing_time":       "playingtime",
    "misc":               "misc",
}

# STAT_TYPES = {
#     "standard":           "stats",
#     "shooting":           "shooting",
#     "passing":            "passing",
#     "passing_types":      "passing_types",
#     "goal_shot_creation": "gca",
#     "defense":            "defense",
#     "possession":         "possession",
#     "playing_time":       "playingtime",
#     "misc":               "misc",
# }

In [ ]:
# ── URL Builder ────────────────────────────────────────────
def get_url(league_key, season, stat_type):
    """Build FBref URL for player stats (note /players/ in path)."""
    league = LEAGUES[league_key]
    cid = league["comp_id"]
    lname = league["name"]
    slug = STAT_TYPES[stat_type]
    return f"https://fbref.com/en/comps/{cid}/{season}/{slug}/players/{season}-{lname}-Stats"


# ── Scraper Function ─────────────────────────────────────────
def scrape_table(url):
    """Fetch page and extract the PLAYER stats table."""
    print(f"    Fetching: {url}")
    try:
        resp = scraper.get(url, timeout=30)
    except Exception as e:
        print(f"    ✗ Request error: {e}")
        return None

    print(f"    Status: {resp.status_code}")

    if resp.status_code == 403:
        print("    ✗ BLOCKED — cookies may have expired! Refresh them.")
        return None
    if resp.status_code != 200:
        print(f"    ✗ Error: {resp.status_code}")
        return None

    soup = BeautifulSoup(resp.text, "lxml")

    # Collect ALL tables — both visible and hidden in comments
    all_tables = []

    # Visible tables
    for t in soup.find_all("table"):
        table_id = t.get("id", "")
        all_tables.append((table_id, t))

    # Tables hidden inside HTML comments
    for comment in soup.find_all(string=lambda t: isinstance(t, Comment)):
        if "<table" in str(comment):
            comment_soup = BeautifulSoup(str(comment), "lxml")
            for t in comment_soup.find_all("table"):
                table_id = t.get("id", "")
                all_tables.append((table_id, t))

    print(f"    Found {len(all_tables)} tables: {[tid for tid, _ in all_tables if tid]}")

    if len(all_tables) == 0:
        print("    ✗ No tables found!")
        return None

    # FBref table IDs follow a pattern:
    #   stats_standard         -> team standard stats
    #   stats_standard_per_match -> team per-match stats  
    #   stats_standard_opponent  -> opponent stats
    #   stats_shooting         -> team shooting stats
    #   etc.
    #
    # The PLAYER stats tables DON'T have these IDs — they typically
    # have IDs like "stats_standard" but appear LATER in the page,
    # OR they have no ID at all.
    #
    # Best approach: find the table that has BOTH "Player" as a 
    # column header AND has many rows (500+), since team tables 
    # only have ~20 rows.

    player_table = None
    for table_id, table in all_tables:
        # Quick check: count rows
        rows = table.find_all("tr")
        if len(rows) < 30:
            continue  # Team tables have ~20 rows, skip them

        # Check if this table has "Player" in its header
        thead = table.find("thead")
        if thead and "Player" in thead.get_text():
            player_table = table
            print(f"    ✓ Found player table (id='{table_id}', {len(rows)} rows)")
            break

    if player_table is None:
        print("    ✗ Could not find player stats table!")
        # Debug: show what tables we found
        for tid, t in all_tables:
            rows = t.find_all("tr")
            thead = t.find("thead")
            has_player = "Player" in thead.get_text() if thead else False
            print(f"      id='{tid}', rows={len(rows)}, has_Player={has_player}")
        return None

    df = pd.read_html(str(player_table), header=[0, 1])[0]

    # Flatten multi-level columns
    df.columns = [
        f"{a}_{b}" if "Unnamed" not in str(a) and a != b else str(b)
        for a, b in df.columns
    ]

    # Remove repeated header rows embedded in data
    if "Rk" in df.columns:
        df = df[df["Rk"] != "Rk"].reset_index(drop=True)

    # Drop the "Matches" column if it exists (just a link)
    if "Matches" in df.columns:
        df = df.drop(columns=["Matches"])

    print(f"    ✓ {df.shape[0]} players x {df.shape[1]} columns")
    return df

In [22]:
# ══════════════════════════════════════════════════════════════
# TEST SCRAPE — Verify player data before full run
# ══════════════════════════════════════════════════════════════
print("=" * 60)
print("TEST: Scraping Premier League 2024-2025 Standard Stats")
print("=" * 60)

test_url = get_url("premier_league", "2024-2025", "standard")
test_df = scrape_table(test_url)

RUN_FULL_SCRAPE = False

if test_df is not None:
    print(f"\n✓ SUCCESS! Got {test_df.shape[0]} players x {test_df.shape[1]} columns")
    print(f"\nFirst 5 players:")
    # Show key columns if they exist
    show_cols = [c for c in ["Player", "Squad", 'Age', "Pos", "MP", "Gls", "Ast"] if c in test_df.columns]
    if show_cols:
        print(test_df[show_cols].head())
    else:
        print(test_df.head())
    print(f"\nAll columns:\n{list(test_df.columns)}")
    RUN_FULL_SCRAPE = True
else:
    print("\n✗ TEST FAILED")
    print("  1. Go to https://fbref.com in Edge")
    print("  2. F12 → Network → refresh → click first request → Headers")
    print("  3. Copy the full 'cookie:' value")
    print("  4. Replace the COOKIES string at the top and re-run")

TEST: Scraping Premier League 2024-2025 Standard Stats
    Fetching: https://fbref.com/en/comps/9/2024-2025/stats/players/2024-2025-Premier-League-Stats
    Status: 200
    Found 3 tables: ['stats_squads_standard_for', 'stats_squads_standard_against', 'stats_standard']
    ✓ Found player table (id='stats_standard', 598 rows)
    ✓ 574 players x 24 columns

✓ SUCCESS! Got 574 players x 24 columns

First 5 players:
              Player        Squad Age Pos
0         Max Aarons  Bournemouth  24  DF
1  Joshua Acheampong      Chelsea  18  DF
2        Tyler Adams  Bournemouth  25  MF
3   Tosin Adarabioyo      Chelsea  26  DF
4      Simon Adingra     Brighton  22  MF

All columns:
['Rk', 'Player', 'Nation', 'Pos', 'Squad', 'Age', 'Born', 'Playing Time_MP', 'Playing Time_Starts', 'Playing Time_Min', 'Playing Time_90s', 'Performance_Gls', 'Performance_Ast', 'Performance_G+A', 'Performance_G-PK', 'Performance_PK', 'Performance_PKatt', 'Performance_CrdY', 'Performance_CrdR', 'Per 90 Minutes_Gls',

C:\Users\natmaw\AppData\Local\Temp\ipykernel_9836\2646353560.py:91: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df = pd.read_html(str(player_table), header=[0, 1])[0]


In [23]:
# ══════════════════════════════════════════════════════════════
# MAIN SCRAPING LOOP — only runs if test passed
# ══════════════════════════════════════════════════════════════
if RUN_FULL_SCRAPE:
    print("\n" + "=" * 60)
    print("FULL SCRAPE: Starting...")
    print(f"Leagues: {list(LEAGUES.keys())}")
    print(f"Seasons: {SEASONS}")
    print(f"Stat types: {list(STAT_TYPES.keys())}")
    total_pages = len(LEAGUES) * len(SEASONS) * len(STAT_TYPES)
    print(f"Total pages to scrape: {total_pages} (~{total_pages * 8 // 60} min)")
    print("=" * 60)

    time.sleep(8)  # Wait after the test request

    for league_key, league_info in LEAGUES.items():
        for season in SEASONS:
            print(f"\n{'━' * 60}")
            print(f"  {league_key.upper()} — {season}")
            print(f"{'━' * 60}")

            # Create folder: data/raw/premier_league/2024-2025/
            season_dir = os.path.join(OUTPUT_DIR, league_key, season)
            os.makedirs(season_dir, exist_ok=True)

            all_stats = {}

            for stat_type in STAT_TYPES:
                print(f"\n  [{stat_type}]")
                url = get_url(league_key, season, stat_type)
                df = scrape_table(url)

                if df is not None:
                    for col in df.columns:
                        try:
                            df[col] = pd.to_numeric(df[col])
                        except (ValueError, TypeError):
                            pass
                    all_stats[stat_type] = df
                else:
                    print(f"    SKIPPED")

                time.sleep(8)

            # Save CSV files into the league/season folder
            for stat_type, df_save in all_stats.items():
                df_save = df_save.copy()
                df_save["league"] = league_key
                df_save["season"] = season
                filename = f"{stat_type}.csv"
                filepath = os.path.join(season_dir, filename)
                df_save.to_csv(filepath, index=False, encoding="utf-8")
                print(f"    ✓ Saved: {league_key}/{season}/{filename} ({len(df_save)} rows)")

            print(f"\n  ✓ ALL SAVED for {league_key} {season}")

    # Summary
    print("\n" + "=" * 60)
    print("SCRAPE COMPLETE!")
    print("=" * 60)
    print(f"\nFolder structure:")
    for league_key in LEAGUES:
        for season in SEASONS:
            season_dir = os.path.join(OUTPUT_DIR, league_key, season)
            if os.path.exists(season_dir):
                files = [f for f in os.listdir(season_dir) if f.endswith(".csv")]
                total_size = sum(os.path.getsize(os.path.join(season_dir, f)) for f in files) / 1024
                print(f"  {league_key}/{season}/ — {len(files)} files ({total_size:.0f} KB)")
                for f in sorted(files):
                    print(f"    {f}")

    print(f"""
Folder structure:
  data/raw/
    premier_league/
      2021-2022/
        standard.csv
        shooting.csv
        passing.csv
        ...
      2022-2023/
        ...
    la_liga/
      2021-2022/
        ...
    bundesliga/
      ...

To load a file:
  df = pd.read_csv('data/raw/premier_league/2023-2024/standard.csv')
""")


FULL SCRAPE: Starting...
Leagues: ['premier_league', 'la_liga', 'bundesliga']
Seasons: ['2021-2022', '2022-2023', '2023-2024', '2024-2025']
Stat types: ['standard', 'goalkeeping', 'shooting', 'playing_time', 'misc']
Total pages to scrape: 60 (~8 min)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  PREMIER_LEAGUE — 2021-2022
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  [standard]
    Fetching: https://fbref.com/en/comps/9/2021-2022/stats/players/2021-2022-Premier-League-Stats
    Status: 200
    Found 3 tables: ['stats_squads_standard_for', 'stats_squads_standard_against', 'stats_standard']
    ✓ Found player table (id='stats_standard', 569 rows)


C:\Users\natmaw\AppData\Local\Temp\ipykernel_9836\2646353560.py:91: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df = pd.read_html(str(player_table), header=[0, 1])[0]


    ✓ 546 players x 24 columns

  [goalkeeping]
    Fetching: https://fbref.com/en/comps/9/2021-2022/keepers/players/2021-2022-Premier-League-Stats
    Status: 200
    Found 3 tables: ['stats_squads_keeper_for', 'stats_squads_keeper_against', 'stats_keeper']
    ✓ Found player table (id='stats_keeper', 45 rows)
    ✓ 42 players x 26 columns


C:\Users\natmaw\AppData\Local\Temp\ipykernel_9836\2646353560.py:91: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df = pd.read_html(str(player_table), header=[0, 1])[0]



  [shooting]
    Fetching: https://fbref.com/en/comps/9/2021-2022/shooting/players/2021-2022-Premier-League-Stats
    Status: 200
    Found 3 tables: ['stats_squads_shooting_for', 'stats_squads_shooting_against', 'stats_shooting']
    ✓ Found player table (id='stats_shooting', 569 rows)


...


  [playing_time]
    Fetching: https://fbref.com/en/comps/20/2024-2025/playingtime/players/2024-2025-Bundesliga-Stats
    Status: 200
    Found 3 tables: ['stats_squads_playing_time_for', 'stats_squads_playing_time_against', 'stats_playing_time']
    ✓ Found player table (id='stats_playing_time', 604 rows)


C:\Users\natmaw\AppData\Local\Temp\ipykernel_9836\2646353560.py:91: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df = pd.read_html(str(player_table), header=[0, 1])[0]


    ✓ 579 players x 24 columns

  [misc]
    Fetching: https://fbref.com/en/comps/20/2024-2025/misc/players/2024-2025-Bundesliga-Stats
    Status: 200
    Found 3 tables: ['stats_squads_misc_for', 'stats_squads_misc_against', 'stats_misc']
    ✓ Found player table (id='stats_misc', 513 rows)


C:\Users\natmaw\AppData\Local\Temp\ipykernel_9836\2646353560.py:91: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df = pd.read_html(str(player_table), header=[0, 1])[0]


    ✓ 492 players x 20 columns
    ✓ Saved: bundesliga/2024-2025/standard.csv (492 rows)
    ✓ Saved: bundesliga/2024-2025/goalkeeping.csv (38 rows)
    ✓ Saved: bundesliga/2024-2025/shooting.csv (492 rows)
    ✓ Saved: bundesliga/2024-2025/playing_time.csv (579 rows)
    ✓ Saved: bundesliga/2024-2025/misc.csv (492 rows)

  ✓ ALL SAVED for bundesliga 2024-2025

SCRAPE COMPLETE!

Folder structure:
  premier_league/2021-2022/ — 5 files (283 KB)
    goalkeeping.csv
    misc.csv
    playing_time.csv
    shooting.csv
    standard.csv
  premier_league/2022-2023/ — 5 files (296 KB)
    goalkeeping.csv
    misc.csv
    playing_time.csv
    shooting.csv
    standard.csv
  premier_league/2023-2024/ — 5 files (302 KB)
    goalkeeping.csv
    misc.csv
    playing_time.csv
    shooting.csv
    standard.csv
  premier_league/2024-2025/ — 5 files (305 KB)
    goalkeeping.csv
    misc.csv
    playing_time.csv
    shooting.csv
    standard.csv
  la_liga/2021-2022/ — 5 files (294 KB)
    goalkeeping.csv
